# Trade Reconciliation Break Assistant

A minimal RAG prototype that retrieves synthetic examples of reconciliation breaks and rules, then explains a likely cause and practical investigation steps. Break detection remains deterministic and outside this notebook; the LLM explains a break supplied by an analyst.

In [ ]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Data Sources and Prototype Choice

### Synthetic break knowledge

Three practical sources for this project are: (1) a synthetic library of paired front-office/custodian trades plus documented reconciliation rules (available immediately and safe for a prototype); (2) internal, de-identified historical breaks and runbooks collected from operations (best realism, but requires approval and anonymization); and (3) public FINRA TRACE trade data for bond-trade examples, supplemented with synthetic custodian-side records because public data is not a paired reconciliation feed.

This notebook uses option 1 so it runs without external access. `data/trade_break_knowledge.json` contains fictional examples and is not operational or customer data.

In [ ]:
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "data").is_dir() and (project_root.parent / "data").is_dir():
    project_root = project_root.parent

data_path = project_root / "data" / "trade_break_knowledge.json"
documents = json.loads(data_path.read_text(encoding="utf-8"))

print(f"Loaded {len(documents)} synthetic break examples from {data_path}")

## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [ ]:
index = Index(
    text_fields=[
        "front_office",
        "custodian",
        "break_summary",
        "likely_root_cause",
        "reconciliation_rule",
        "recommended_actions",
    ],
    keyword_fields=["case_id", "break_category", "asset_class", "severity", "status"],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [ ]:
def search(query):
    return index.search(query, num_results=5)

In [ ]:
instructions = """
You are a trade-reconciliation assistant for operations analysts. The break
has already been detected by deterministic reconciliation code; do not claim
to run or change that matching logic. Use the retrieved synthetic case notes
and reconciliation rules to explain the most likely root cause, the evidence
that supports it, and prioritized investigation/next steps. Distinguish facts
from hypotheses, avoid presenting a possible cause as certain, and do not
invent trade details, policies, or approvals. If the context is insufficient,
say what is unknown and what evidence to request. Keep the answer concise and
use these headings: Likely cause, Why it fits, Next steps, and Caveats.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [ ]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [ ]:
print(rag(
    "For equity trade EQ-1042, front office booked a buy of 10,000 ACME shares, "
    "but the custodian shows 9,750. Trade date, price, and settlement date agree. "
    "What is the likely cause and what should I check next?"
))